# Suite TXN — Transactions: mutation expressions over frames, and what they are built from

A transaction is a mutation expression over a stream of frames ([FEA-027: mutations as queries](../../../mbse-basis/docs/features/FEA-027.md); object identity
85 to 97 in [FEA-030: transactions](../../../mbse-basis/docs/features/FEA-030.md)): its terms are reference objects of the `Mutations` schemas, linked by relations, each term's
pattern a `Schemas.OfPattern` built through its link, and a term chosen through a kind union, `Mutations.OfTerm`. A
pattern's `spec` says what must hold of a frame, written in the builders' notation over the frame's symbols and recorded
as mutations. A keyed adjacency takes a mapping in place of the peer shorthand (FRAMEWORK.md, Resolved).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, JSON as Json, Paths, Plain, Proposals, Proxies, References, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, instance_problems, map, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by name; crm.Note, counted. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject, Note: S.OfObject,
      Meets: S.OfRelation, Kind: S.OfUnion, Picks: S.OfRelation, Pins: S.OfRelation, Odd: S.OfObject, Odds: S.OfRelation })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
      ]).adjacencies({
        meets: (t: any) => t.relation([_.Meets]).me("a"),
        trio: (t: any) => t.relation([S.OfRelation((r: any) => r.links([ // no name of its own
          (t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]))]).me("a"),
      }),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        picks: (t: any) => t.relation([_.Picks]).me("owner"),
        pins: (t: any) => t.relation([_.Pins]).me("note"),
        odds: (t: any) => t.relation([_.Odds]).me("note"),
      }),
      (_: any) => _.Meets.name("crm.Meets").links([(t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]),
      (_: any) => _.Kind.name("crm.Kind").branches([
        (t: any) => t.name("person").schema([_.Person]), (t: any) => t.name("note").schema([_.Note])]),
      (_: any) => _.Picks.name("crm.Picks").links([(t: any) => t.name("owner"), (t: any) => t.name("pick").schema([_.Kind])])
        .properties({ rank: S.OfInt }),
      (_: any) => _.Pins.name("crm.Pins").links([(t: any) => t.name("note"), (t: any) => t.name("book").schema([_.Book])])
        .properties({ at: S.OfInt }),
      (_: any) => _.Odd.name("crm.Odd").ref(true).adjacencies([(t: any) => t.name("x").me("a")]), // no relation yet
      (_: any) => _.Odds.name("crm.Odds").links([(t: any) => t.name("note"), (t: any) => t.name("odd").schema([_.Odd])])
        .properties({ k: S.OfInt }),
    ])]))]);
}

const store: any = new B.Store();
const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
store.transaction(crm);
const [Person, Knows] = [store.registered("crm.Person"), store.registered("crm.Knows")];
const [Book, Note] = [store.registered("crm.Book"), store.registered("crm.Note")];

## TXN-01 · A kind union: a link's schema a union of reference object schemas, each branch a kind built through it

In [ ]:
{
  const Mutations = store.Mutations;
  const OfTerm = Mutations.OfTerm.as_schema();
  const kinds = S._kinds(OfTerm) as Map<string, unknown>;
  assert(same([...kinds.keys()], ["map", "reduce", "partition", "update", "upsert"])
    && [...kinds.values()].every((kind, i) => kind === [Mutations.OfMap, Mutations.OfReduce, Mutations.OfPartition,
      Mutations.OfUpdate, Mutations.OfUpsert][i].as_schema()));
  assert(S._kinds(Person) === null && S._kinds(store.Schemas.OfStr.as_schema()) === null);
  assert(store.names().filter((name: string) => name.startsWith("Mutations.")).every((name: string) => store.registered(name).validate().length === 0));
  store.transaction(() => {
    const mixed = store.Schemas.OfUnion((u: any) => u.branches([(t: any) => t.name("p").schema([Person]),
      (t: any) => t.name("v").schema([store.Schemas.OfObject((o: any) => o)])]));
    assert(same(store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b").schema([mixed])])).validate(),
      ["link 'b': a kind union's branches are reference object schemas"]));
  });
  // Building through the link: a branch, called, builds an object of its kind; the peer shorthand's visitor and an entry
  // builder each have the branches as members.
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-1").mutations([(mu: any) => mu.upsert((u: any) => (
    u.key("k").then([(w: any) => w.update((d: any) => d)])))]));
  const root = tx.mutations[0].term;
  assert(root.as_schema() === Mutations.OfUpsert.as_schema() && root.key.get() === "k");
  assert(root.then[0].next.as_schema() === Mutations.OfUpdate.as_schema());
  const other = store.Mutations.OfTransaction((t: any) => t.uid("t-2").mutations((a: any) => a.insert((e: any) => e.map((m: any) => m))));
  assert(other.mutations[0].term.as_schema() === Mutations.OfMap.as_schema());
  const kept = store.Mutations.OfTransaction((t: any) => t.uid("t-3").mutations((a: any) => a.insert((e: any) => e.update(root.then[0].next))));
  assert(kept.mutations[0].term === root.then[0].next); // a kind given an object links it
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.uid])), // a member of no kind: which object?
    "link 'term' is a kind union: build through one of its kinds, map, reduce, partition, update, upsert");
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => e.term((x: any) => x)))),
    "link 'term' is a kind union: build through one of its kinds");
  const seen: string[] = [];
  store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => (seen.push(repr(e.upsert)), e.upsert((u: any) => u)))));
  assert(same(seen, ["<kind of link 'term'>"]));
}

## TXN-02 · Mappings for keyed adjacencies: each item an entry, its key the one property, its value a callback or given to the one member that takes it

In [ ]:
{
  assert(same([...Person.properties.keys()], ["name", "age"]) && Person.properties.get("age").schema === Int); // the schema builders'
  const { Tagged } = store.transaction(() => {
    const Tagged = store.Schemas.OfObject((o: any) => o.properties({ tag: (t: any) => t.schema([Str]).description("a label") }));
    assert(Tagged.properties.get("tag").description === "a label");
    raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas({ n: Int })),
      "'schemas' takes a mapping only where its relation has two links and one property");
    raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: "text" })), // its description and its naming pattern
      "more than one member of the entry takes 'text': give a callback");
    raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: 3n })), "no member of the entry takes 3: give a callback");
    return { Tagged };
  });
  const rootModule = store.module();
  store.transaction(() => {
    store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ Tagged })); // the link alone takes a schema
  });
  assert(rootModule.schemas.get("Tagged") === Tagged);
  const book = store.singleton("crm.TheBook");
  store.transaction(() => {
    store.crm.Book(book, (b: any) => b.contacts({ ada: (e: any) => e.name("Ada"), bob: (e: any) => e.name("Bob").age(36n) }));
  });
  const [ada, bob] = [book.contacts({ key: "ada" }), book.contacts({ key: "bob" })]; // the proxies': each a contact, by its key
  assert(equal([...book.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ada", "Ada"], ["bob", "Bob"]]) && bob.age.get() === 36n);
  store.transaction(() => {
    store.crm.Person(ada, (c: any) => c.knows(new Map([[2020n, (e: any) => e.known(bob)]])));
  });
  assert(equal([...ada.knows].map((e: any) => [e.since.get(), e.known]), [[2020n, bob]]));
  store.transaction(() => {
    raises(TypeError, () => store.crm.Person(ada, (c: any) => c.knows(new Map([[2021n, bob]]))), // the link, and the linked person's own 'knows'
      "more than one member of the entry takes <crm.Person>: give a callback");
    raises(TypeError, () => store.crm.Book(book, (b: any) => b.contacts({ x: 1.5 })), "no member of the entry takes 1.5: give a callback");
  });
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations({ x: (e: any) => e })),
    "adjacency 'mutations' takes a mapping only where its relation has two links and one property");
  store.transaction(() => {
    raises(TypeError, () => store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ x: "x" })), "no member of the entry takes 'x': give a callback");
    raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas("x")),
      "'schemas' takes an iterable of what it links, or of entries, a mapping, or a callback");
  });
  // A link of a kind union, or of a schema it declares, takes an object of it; a callback chooses a kind.
  const { note } = store.transaction(() => {
    const note = store.crm.Note((n: any) => n.picks(new Map<bigint, unknown>([[1n, ada], [2n, (e: any) => e.note((m: any) => m.text("aside"))]]))
      .pins(new Map([[3n, book]])));
    return { note };
  });
  assert(equal([...note.picks].map((e: any) => [e.rank.get(), e.pick.as_schema().name]), [[1n, "crm.Person"], [2n, "crm.Note"]]));
  assert(equal([...note.pins].map((e: any) => [e.at.get(), e.book]), [[3n, book]]));
  store.transaction(() => {
    raises(TypeError, () => store.crm.Note((n: any) => n.odds(new Map([[1n, "zz"]]))), // an adjacency with no relation yet takes nothing
      "no member of the entry takes 'zz': give a callback");
  });
  const outside = new B.OfObject.Builder().properties({ n: (t: any) => t.description("d") }).create(); // outside a store too
  assert(outside.properties.get("n").description === "d");
}

## TXN-03 · A transaction's expression: reference objects linked by relations, each term's pattern a schema object built through its link and held while its term is

In [ ]:
{
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-9").mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ ada: Person }).description("whom")]))]));
  const pattern = tx.mutations[0].term.exists[0].pattern;
  assert(pattern instanceof S.OfPattern.Data && same([...pattern.symbols.keys()], ["ada"]) && pattern.description === "whom");
  assert(store.holds(pattern) && store.holds(pattern.symbols.get("ada"))); // weakly: it goes with its transaction (Decided, 94)
  raises(LookupError, () => Paths.of(pattern), "a pattern with no name of its own");
  /** crm.Edit, a pattern over crm.Person, which `crm` installed. */
  const edits = (txn: any): any => txn.uid("edit/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Edit: txn.Schemas.OfPattern })
    .spec([(_: any) => _.Edit.name("crm.Edit").symbols({ ada: txn.crm.Person })])]))]);
  store.transaction(edits);
  const Edit = store.registered("crm.Edit");
  store.Mutations.OfUpdate((u: any) => u.apply([(p: any) => p.pattern(Edit).description("edits")])); // linked, then updated
  assert(Edit.description === "edits");
  assert(store._meta_builder("crm.Person") === null && store._meta_builder("Schemas.Nope") === null);
  assert(repr(store._meta_builder("Schemas.OfPattern")) === "<builder of a pattern>");
  assert(store._meta_builder_class(Edit) === B.PatternBuilder);
}

## TXN-04 · A spec: written in the builders' notation over the frame's symbols, recorded as mutations, by symbol; an adjacency's entries inserted, removed or cleared

In [ ]:
{
  store.transaction(() => { // the patterns its specs are written in
    const ref = (name: string): any => ({ ref: { symbol: name } });
    const pathOf = (target: unknown): any => new Map([["ref", new Map([["path", Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(target))]])]]);
    // (a path is a Map already: `map` keeps it)
    const link = (name: string, target: unknown): any => ({ as_link: { name, mutations: [{ set: target }] } });
    const prop = (name: string, value: unknown): any => ({ as_property: { name, mutations: [{ set: { value } }] } });
    const entries = (name: string, ...items: unknown[][]): any => ({ as_adjacency: { name, mutations: [{ insert: {
      entries: items.map((item) => ({ mutations: item })) } }] } });

    const Meet = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
      (_: any) => _.bob.name("Bob").knows([_.ada]),
      (_: any) => _.ada.name("Ada").age(null).knows([(e: any) => e.known(_.bob).since(2020n)]),
    ]));
    assert(equal(Meet.spec, map([
      { bob: [prop("name", "Bob"), entries("knows", [link("known", ref("ada"))])] },
      { ada: [prop("name", "Ada"), { as_property: { name: "age", mutations: [{ clear: {} }] } },
        entries("knows", [link("known", ref("bob")), prop("since", 2020n)])] },
    ])));
    const book = store.singleton("crm.TheBook");
    const Owned = store.Schemas.OfPattern((p: any) => p.symbols({ bob: Person }).spec( // one callback; a path; a mapping
      (_: any) => _.bob.book({ bob: book }).knows((a: any) => a.insert((e: any) => e.known((n: any) => n.name("Dan")).since(1n)))));
    assert(equal(Owned.spec, map([{ bob: [
      entries("book", [link("book", { ref: pathOf(book).get("ref") }), prop("key", "bob")]),
      entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 1n)]),
    ] }])));
    const Defines = store.Schemas.OfPattern((p: any) => p.symbols({ P: store.Schemas.OfObject }).spec([
      (_: any) => _.P.name("crm.Pet").ref(true).properties({ name: Str, legs: (t: any) => t.schema([Int]) }),
    ]));
    const propertySchema = (schema: unknown): any => ({ def: { schema: "Schemas.OfProperty", mutations: [
      entries("schema", [link("schema", { ref: pathOf(schema).get("ref") })])] } });
    assert(equal(Defines.spec, map([{ P: [prop("name", "crm.Pet"), prop("ref", true), entries(
      "properties", [link("property", propertySchema(Str)), prop("name", "name")],
      [link("property", propertySchema(Int)), prop("name", "legs")])] }])));
    const Terms = store.Schemas.OfPattern((p: any) => p.symbols({ t: store.Mutations.OfTransaction }).spec([
      (_: any) => _.t.uid("u").mutations([(e: any) => e.upsert((u: any) => u.key("k"))]), // a kind union: a kind defined
      (_: any) => _.t.mutations([(e: any) => e.term(_.t)]),
    ]));
    assert(equal(Terms.spec[0], map({ t: [prop("uid", "u"), entries("mutations", [link("term", { def: {
      schema: "Mutations.OfUpsert", mutations: [prop("key", "k")] } })])] })));
    assert(equal(Terms.spec[1], map({ t: [entries("mutations", [link("term", ref("t"))])] })));
    const Removes = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([ // as a builder's adjacency
      (_: any) => _.ada.knows((a: any) => a.remove((e: any) => e.known(_.bob).since(2020n)).clear()),
    ]));
    assert(equal(Removes.spec, [map({ ada: [
      { as_adjacency: { name: "knows", mutations: [{ remove: { entries: [{ mutations: [
        link("known", ref("bob")), prop("since", 2020n)] }] } }] } },
      { as_adjacency: { name: "knows", mutations: [{ clear: {} }] } },
    ] })]));
    // Reading through the frame, and what a spec refuses.
    const seen: unknown[] = [];
    store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([
      (_: any) => seen.push(repr(_), repr(_.ada), _.ada.name.get(), _.ada.as_schema() === Person,
        _.nope.as_schema(), _.ada.nope.as_schema(), _.ada.knows.as_schema().name,
        "_hidden" in _, "_hidden" in _.ada, "_hidden" in _.ada.name),
      (_: any) => _.ada.knows((a: any) => a.insert((e: any) => (seen.push(
        e.as_schema() === Knows.entry, "_hidden" in e, e.nope.as_schema()), e.known(_.ada)))),
      (_: any) => _.ada.knows([(e: any) => e.known((n: any) => { seen.push(repr(n)); })]),
    ]));
    assert(same(seen, ["<frame>", "<spec of ada>", null, true, null, null, "knows", false, false, false, true, false, null,
      "<spec of an object>"]));
    const Given = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
      (_: any) => _.bob.knows(new Map([[2020n, "Dan"]])).book({ b: _.ada }), // a property of the linked object; an adjacency of it
      (_: any) => _.bob.knows([(e: any) => e.known((n: any) => n.name("Eve")).age(3n)]), // the object it defines, written on
    ]));
    assert(equal(Given.spec[0], map({ bob: [
      entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 2020n)]),
      entries("book", [link("book", { def: { schema: "crm.Book", mutations: [entries("contacts", [link("contact", ref("ada"))])] } }),
        prop("key", "b")])] })));
    assert(equal(Given.spec[1], map({ bob: [entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [
      prop("name", "Eve"), prop("age", 3n)] } })])] })));
    const note = store.crm.Note((n: any) => n);
    for (const [spec, error, says] of [
      [(_: any) => _.x.name("x"), TypeError, "symbol 'x' has neither a schema nor a reference: a symbol has exactly one"],
      [(_: any) => _.ada.knows(3n), TypeError, "adjacency 'knows' takes an iterable of entries or of what it links, a mapping"],
      [(_: any) => _.ada.book({ a: 1.5 }), TypeError, "no member of the entry takes 1.5: give a callback"],
      [(_: any) => _.ada.knows([(e: any) => e.known(_.ada).name("Z")]), ValueError,
        "link 'known' links an object a spec does not define: write its members in its own spec"],
      [(_: any) => _.ada.knows([(e: any) => e.known(note)]), ValueError,
        "a counted 'crm.Note' has no path: a spec refers to it through a symbol"],
      [(_: any) => _.ada.meets([(e: any) => e]), TypeError,
        "adjacency 'meets': entries of crm.Meets have more than two links; give each entry's links"],
      [(_: any) => _.ada.trio([(e: any) => e]), TypeError, "adjacency 'trio': entries of its relation have more than two links"],
      [(_: any) => _.ada.knows(new Map([[2020n, _.ada]])), TypeError, "more than one member of the entry takes <spec of ada>: give a callback"],
      [(_: any) => _.ada.meets(new Map([[1n, _.ada]])), TypeError,
        "adjacency 'meets' takes a mapping only where its relation has two links and one property"],
      [(_: any) => _.ada.knows([(e: any) => e.known((n: any) => n.knows([(f: any) => f.known(n)]))]), ValueError,
        "an object a spec defines is linked where it is defined"],
      [(_: any) => _.t.mutations([(e: any) => e.term((x: any) => x)]), TypeError,
        "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
      [(_: any) => _.t.mutations([(e: any) => e.uid("u")]), TypeError,
        "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
    ] as [(_: any) => unknown, Function, string][]) {
      raises(error, () => store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, x: (t: any) => t,
        t: store.Mutations.OfTransaction }).spec([spec])), says);
    }
    const entry: any[] = [];
    store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([(_: any) => _.ada.knows([(e: any) => (entry.push(e), e.known(_.ada))])]));
    raises(AttributeError, () => { entry[0].since = 1n; }); // a member is written by calling it
    raises(TypeError, () => new B.PatternBuilder().spec([(_: any) => _]), "a spec is written in a store, whose schemas its symbols name");
    raises(ReadOnlyError, () => new B.PatternBuilder(Meet, false, store).spec([(_: any) => _]));
  });
}

## TXN-05 · A transaction's proposals: data written apart from the store until it commits, dropped where it raises; a schema object that existed changed at its commit

In [ ]:
{
  const store5: any = new B.Store();
  const [Int5, Str5] = [store5.Schemas.OfInt, store5.Schemas.OfStr];
  store5.transaction(crm);
  const [Person5, Book5, Note5] = [store5.registered("crm.Person"), store5.registered("crm.Book"), store5.registered("crm.Note")];
  const book5 = store5.singleton("crm.TheBook");
  let ada5: any = null;
  let note5: any = null;
  let temp5: any = null;
  store5._transacting(() => {
    store5.crm.Book(book5, (b: any) => b.contacts({ ada: (e: any) => e.name("Ada") }));
    ada5 = book5.contacts({ key: "ada" });
    note5 = store5.crm.Note((n: any) => n.text("x"));
    store5.crm.Note(note5, (n: any) => n.text("y"));
    assert(ada5.name.get() === "Ada" && note5.text.get() === "y"); // read through its proposals
  });
  const keys = (): unknown[] => [...book5.contacts].map((e: any) => e.key.get());
  assert(same(keys(), ["ada"]) && note5.text.get() === "y"); // committed
  raises(ValueError, () => store5._transacting(() => {
    store5.crm.Book(book5, (b: any) => b.contacts({ bob: (e: any) => e.name("Bob") }));
    store5.crm.Person(ada5, (c: any) => c.name("Ada L."));
    store5.Schemas.OfObject((o: any) => o.name("crm.Gone").ref(true));
    temp5 = store5.Schemas.OfObject((o: any) => o.ref(true)); // no name of its own, then one
    store5.Schemas.OfObject(temp5, (o: any) => o.name("crm.Temp"));
    throw new ValueError("stop");
  }), "stop");
  assert(same(keys(), ["ada"]) && ada5.name.get() === "Ada"); // dropped
  assert(!store5.names().includes("crm.Gone") && !store5.module("crm").schemas.has("Gone") && !store5.holds(temp5));
  store5._transacting(() => { // a schema object that existed: changed when the transaction commits
    store5.Schemas.OfObject(Note5, (o: any) => o.properties({ tag: Str5 }));
    assert(!Note5.properties.has("tag"));
    store5.Schemas.OfObject(Note5, (o: any) => o.properties({ mood: Str5 })); // from what the one before made
    const Fresh = store5.Schemas.OfObject((o: any) => o.name("crm.Fresh").ref(true));
    store5.Schemas.OfObject(Fresh, (o: any) => o.properties({ n: Int5 })); // one it made: changed at once
    assert(same([...Fresh.properties.keys()], ["n"]));
  });
  assert(same([...Note5.properties.keys()], ["text", "tag", "mood"]));
  raises(ValueError, () => store5._transacting(() => { // refused when it commits: what it wrote is undone, the schemas' and the data's
    store5.Schemas.OfObject(Person5, (o: any) => o.description("people"));
    store5.crm.Note(note5, (n: any) => n.text("z"));
    store5.Schemas.OfObject(Note5, (o: any) => o.name("crm.Book"));
  }), "schema 'crm.Book' is already registered");
  assert(Person5.description === null && Note5.name === "crm.Note" && note5.text.get() === "y");
  assert(store5.registered("crm.Note") === Note5 && store5.registered("crm.Book") === Book5);
  raises(ValueError, () => store5._transacting(() => store5._transacting(() => null)),
    "a transaction is open: transactions do not nest");
}

## TXN-06 · store.transaction: a fixture upserted in one transaction, its frames given back; run again, it finds what it made, an owned object by its spec, a counted one by its frame, key and symbol

In [ ]:
{
  const store6: any = new B.Store();
  store6.transaction(crm);
  const [Person6, Book6, Note6] = [store6.registered("crm.Person"), store6.registered("crm.Book"), store6.registered("crm.Note")];
  const book6 = store6.singleton("crm.TheBook");
  const fixture = (t: any): any => t.uid("fixture").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { ada: Person6, bob: Person6, book: Book6, note: Note6 }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada), bob: (e: any) => e.contact(_.bob) }),
    (_: any) => _.ada.name("Ada").knows([(e: any) => e.known(_.bob).since(2020n)]),
    (_: any) => _.bob.name("Bob").age(null),
    (_: any) => _.note.text("hi").pins(new Map([[1n, _.book]])).odds([(e: any) => e.odd((o: any) => o).k(7n)]), // an Odd defined
  ])]))]);
  const frames = store6.transaction(fixture);
  assert(frames.length === 1 && same([...frames[0].keys()], ["ada", "bob", "book", "note"]) && frames[0].get("book") === book6);
  const [ada6, bob6, note6] = ["ada", "bob", "note"].map((name) => frames[0].get(name));
  assert(equal([...book6.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ada", "Ada"], ["bob", "Bob"]]));
  assert(equal([...ada6.knows].map((e: any) => [e.known === bob6, e.since.get()]), [[true, 2020n]]));
  assert(equal([...note6.pins].map((e: any) => [e.book === book6, e.at.get()]), [[true, 1n]]) && note6.text.get() === "hi");
  const odd6 = note6.odds[0].odd;
  assert(odd6.as_schema() === store6.registered("crm.Odd") && note6.odds[0].k.get() === 7n);
  const again = store6.transaction(fixture); // found, not made again: nothing to write
  assert(same(["ada", "bob", "note"].map((name) => again[0].get(name)), [ada6, bob6, note6]) && book6.contacts.length === 2);
  assert(ada6.knows.length === 1 && note6.pins.length === 1 && same([...note6.odds].map((e: any) => e.odd), [odd6]));
  const byPath = store6.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person6 }).spec([(_: any) => _.x.book({ ada: book6 })])]))])); // the book by its path
  assert(byPath.length === 1 && byPath[0].size === 1 && byPath[0].get("x") === ada6);
  const built = store6.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.key("other").exists([
    (p: any) => p.symbols({ note: Note6 }).spec([(_: any) => _.note.text("hi")])]))])); // built first, then executed
  const other = store6.transaction(built)[0].get("note");
  assert(other !== note6 && store6.transaction(built)[0].get("note") === other); // a key of its own: another note, kept, though one with its text exists (102)
  const none = store6.transaction((t: any) => t);
  assert(none.length === 1 && none[0].size === 0); // no term: the one empty frame
  const singleton = store6.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ book: Book6 })]))]));
  assert(singleton.length === 1 && singleton[0].get("book") === book6); // a singleton's symbol binds it
}

## TXN-07 · Map and update: a map binds each object its spec matches, one frame each, the frames before it multiplied; an update writes its spec over each frame

In [ ]:
const store7: any = new B.Store();
store7.transaction(crm);
const [Person7, Book7, Note7] = [store7.registered("crm.Person"), store7.registered("crm.Book"), store7.registered("crm.Note")];
const book7 = store7.singleton("crm.TheBook");
store7.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
  { a: Person7, b: Person7, c: Person7, book: Book7 }).spec([
  (_: any) => _.book.contacts({ a: (e: any) => e.contact(_.a), b: (e: any) => e.contact(_.b), c: (e: any) => e.contact(_.c) }),
  (_: any) => _.a.name("Ada").age(36n).knows([(e: any) => e.known(_.b).since(2020n), (e: any) => e.known(_.c).since(2021n)]),
  (_: any) => _.b.name("Bob").age(36n).knows([_.c]),
  (_: any) => _.c.name("Cy"),
])]))]));

function mapped(specs: unknown[], symbols: any = null, then: unknown = null): any[] {
  const term = (m: any): any => m.symbols([(p: any) => p.symbols(symbols ?? { x: Person7, y: Person7 }).spec(specs)]);
  return store7.transaction((t: any) => t.mutations([(mu: any) => mu.map(
    then === null ? term : (m: any) => term(m).then([then]))]));
}

const names = (frames: any[]): string[][] => frames.map((frame) => [...frame.values()].map((target: any) => target.name.get()));
{
  assert(equal(names(mapped([(_: any) => _.x.knows([_.y])])), [["Ada", "Bob"], ["Ada", "Cy"], ["Bob", "Cy"]]));
  assert(equal(names(mapped([(_: any) => _.x.knows([(e: any) => e.known((k: any) => k.name("Bob").knows([_.y]))])])),
    [["Ada", "Cy"]])); // through one
  assert(equal(names(mapped([(_: any) => _.x.knows([(e: any) => e.known(_.y).since(2021n)])])), [["Ada", "Cy"]]));
  assert(equal(names(mapped([(_: any) => _.x.age(36n)], { x: Person7 })), [["Ada"], ["Bob"]]));
  const about = mapped([(_: any) => _.x.name("Ada")], { x: Person7 }, (w: any) => w.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ x: Person7, n: Note7 }).spec([(_: any) => _.n.text("about Ada")])])));
  assert(equal(about.map((f: any) => [f.get("x").name.get(), f.get("n").text.get()]), [["Ada", "about Ada"]])); // x as the map bound it
  assert(equal(names(mapped([(_: any) => _.x.age(null)], { x: Person7 })), [["Cy"]])); // a property cleared: none held
  assert(equal(names(mapped([(_: any) => _.x.name("Nobody")], { x: Person7 })), []));
  assert(mapped([], { x: Person7, y: Person7 }).length === 9); // no spec: the cross product
  assert(equal(mapped([(_: any) => _.x.book({ b: _.y })], { x: Person7, y: Book7 }).map((f: any) => [f.get("x").name.get(),
    f.get("y") === book7]), [["Bob", true]])); // an entry's property, through a mapping
  store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person7 }).spec([(_: any) => _.x.age(36n)])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
    { x: Person7 }).spec([(_: any) => _.x.age(37n)])]))]))]));
  assert(equal([...store7.extent("crm.Person")].map((p: any) => [p.name.get(), store7.member(p, "age")])
    .sort((x: any, y: any) => (x[0] < y[0] ? -1 : 1)), [["Ada", 37n], ["Bob", 37n], ["Cy", null]]));
  store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person7, bk: Book7 }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person7, bk: Book7 }).spec([(_: any) => _.x.age(null).book({ robert: _.bk })])]))]))]));
  const bob7 = book7.contacts({ key: "robert" }); // its one owning entry replaced: its key now "robert"
  assert(bob7.name.get() === "Bob" && store7.member(bob7, "age") === null
    && same([...book7.contacts].map((e: any) => e.key.get()), ["a", "c", "robert"]));
}
const tasks7b = (txn: any): any => {
  const S = txn.Schemas;
  return txn.uid("tasks/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ InTasks: S.OfRelation, Directory: S.OfObject, Task: S.OfObject })
    .spec([
      (_: any) => _.InTasks.name("work.InTasks").links([(t: any) => t.name("directory"), (t: any) => t.name("task")])
        .properties({ id: S.OfStr }).uniques([["directory", "id"], ["task"]]),
      (_: any) => _.Directory.name("work.TaskDirectory").ref(true).singleton("work.Tasks")
        .adjacencies({ tasks: (a: any) => a.relation([_.InTasks]).me("directory") }),
      (_: any) => _.Task.name("work.Task").ref(true).properties({ title: S.OfStr })
        .adjacencies([(t: any) => t.name("directory").relation([_.InTasks]).me("task").owner(true)]),
    ])]))]);
};
const store7b: any = new B.Store();
store7b.transaction(tasks7b);
store7b.transaction((t: any) => t.uid("seed").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
  .symbols({ d: store7b.work.TaskDirectory, a: store7b.work.Task, b: store7b.work.Task }).spec([
    (_: any) => _.d.tasks({ "TSK-001": (e: any) => e.task(_.a), "TSK-002": (e: any) => e.task(_.b) }),
    (_: any) => _.a.title("one"), (_: any) => _.b.title("two")])]))]));
/** The titles a map over the directory's entries binds, the entry its spec gives. */
const owned = (entry: (_: any) => any): string[] => store7b.transaction((t: any) => t.mutations([(mu: any) => mu.map(
  (m: any) => m.symbols([(p: any) => p
    .symbols({ x: References.of(store7b, store7b.singleton("work.Tasks")), y: store7b.work.Task })
    .spec([(_: any) => _.x.tasks([entry(_)])])]))])).map((frame: any) => String(frame.get("y").title));
assert(equal(owned((_: any) => _.y), ["one", "two"])); // an owned object, its owner's entries' key left open: one frame each (65)
assert(equal(owned((_: any) => (e: any) => e.task(_.y)), ["one", "two"]));
assert(equal(owned((_: any) => (e: any) => e.task(_.y).id("TSK-002")), ["two"])); // its key given: the one entry
assert(equal(owned((_: any) => (e: any) => e.task(_.y).id("TSK-009")), [])); // none

## TXN-08 · Reduce and partition: a reduce drops symbols, keeping each frame left once; a partition runs its first, last or every branch, or none where it has no branch, and joins what they end with

In [ ]:
{
  const pairs = (m: any): any => m.symbols([(p: any) => p.symbols({ x: Person7, y: Person7 }).spec([(_: any) => _.x.knows([_.y])])]);
  const dropping = (dropped: string[]) => (r: any): any => r.symbols([(c: any) => c.pattern([(p: any) => p.symbols(
    { x: Person7, y: Person7 })]).symbols(dropped)]);
  const run = (then: unknown): any[] => store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => pairs(m).then([then]))]));
  assert(equal(names(run((w: any) => w.reduce(dropping(["y"])))), [["Ada"], ["Bob"]])); // three frames, two left
  assert(equal(names(run((w: any) => w.reduce(dropping(["x"])))), [["Cy"], ["Bob"]])); // in the order the store reaches them
  const labelled = (text: string) => (e: any): any => e.branch((b: any) => b.then([(w: any) => w.upsert((u: any) => u.key(text).exists([
    (p: any) => p.symbols({ n: Note7 }).spec([(_: any) => _.n.text(text)])]))]));
  const arms = (p: any): any => p.branches([labelled("one"), labelled("two")]);
  const partitioned = ({ first = false, last = false, branches = arms, then = null }: { first?: boolean, last?: boolean,
    branches?: (p: any) => any, then?: unknown } = {}): any[] => run((w: any) => w.partition((p: any) => {
    const q = branches(p.first(first).last(last));
    return then === null ? q : q.then([then]);
  }));
  const texts = (frames: any[]): string[] => frames.map((frame) => frame.get("n").text.get());
  const times = (text: string, n: number): string[] => Array(n).fill(text);
  assert(same(texts(partitioned()), [...times("one", 3), ...times("two", 3)])); // every branch, each frame through each
  assert(same(texts(partitioned({ first: true })), times("one", 3)) && same(texts(partitioned({ last: true })), times("two", 3)));
  const otherwise = (p: any): any => p.none([(w: any) => w.reduce(dropping(["y"]))]);
  assert(equal(names(partitioned({ branches: otherwise })), [["Ada"], ["Bob"]])); // no branch: its none
  assert(partitioned({ branches: (p: any) => p }).length === 0); // neither: its frames end
  const joined = partitioned({ then: (w: any) => w.reduce(dropping(["y", "n"])) }); // what the branches end with, joined, then on
  assert(equal(names(joined), [["Ada"], ["Bob"]]));
  assert(run((w: any) => w.partition((p: any) => p.branches([(e: any) => e.branch((b: any) => b)]))).length === 3); // a branch with no then: as it is
}

## TXN-09 · What a transaction refuses before it runs (a symbol no term before binds, an expression, a pattern's variables, a symbol of no reference object schema), and a transaction that raises leaving the store as it was

In [ ]:
{
  const store9: any = new B.Store();
  store9.transaction(crm);
  const [Person9, Book9] = [store9.registered("crm.Person"), store9.registered("crm.Book")];
  const Int9 = store9.Schemas.OfInt;
  /** crm.True, an expression a pattern requires. */
  const truth = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("true/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Exprs: S.OfExpression })
      .spec([(_: any) => _.Exprs.name("crm.True").pattern([(p: any) => p]).result([S.OfInt])])]))]);
  };
  store9.transaction(truth);
  const Exprs = store9.registered("crm.True");
  const person = (p: any): any => p.symbols({ x: Person9 });
  const refused = (term: unknown, says: string): void =>
    raises(ValueError, () => store9.transaction((t: any) => t.mutations([term])), says);
  refused((mu: any) => mu.update((u: any) => u.apply([person])), "symbol 'x' of an update is bound by no term before it");
  refused((mu: any) => mu.reduce((r: any) => r.symbols([(c: any) => c.pattern([person]).symbols(["x"])])),
    "symbol 'x' of a reduce is bound by no term before it");
  refused((mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person9 }).requires([Exprs])])),
    "a map's pattern requires constraints: transactions evaluate no expressions yet");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.variables({ v: Int9 })])), "an upsert's pattern has variables");
  refused((mu: any) => mu.map((m: any) => m.variables([person])), "a map of variables");
  refused((mu: any) => mu.map((m: any) => m.symbols([person]).then([(w: any) => w.reduce((r: any) => r.symbols([
    (c: any) => c.pattern([person]).symbols(["x"]).suffixes({ current: "_a", next: "_b" })]))])), "a reduce that collects");
  refused((mu: any) => mu.partition((p: any) => p.first(true).last(true)),
    "a partition dispatches to its first matching branch or its last, not both");
  refused((mu: any) => mu.partition((p: any) => p.branches([(e: any) => e.branch((b: any) => b.requires([Exprs]))])),
    "a branch requires constraints");
  refused((mu: any) => mu.map((m: any) => m), "a map takes one pattern in 'symbols', not 0");
  refused((mu: any) => mu.reduce((r: any) => r), "a reduce takes one collector in 'symbols', not 0");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ s: Int9 })])),
    "symbol 's' binds no reference object of one schema");
  const given9 = References.of(store9, store9.singleton("crm.TheBook")); // a symbol has exactly one of a schema and a reference
  refused((mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols([(s: any) => s.name("s").schema([Book9]).reference(
    given9)])])), "symbol 's' has both a schema and a reference: a symbol has exactly one");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols([(s: any) => s.name("s")])])),
    "symbol 's' has neither a schema nor a reference: a symbol has exactly one");
  refused((mu: any) => mu.map((m: any) => m.symbols([person]).then([(w: any) => w.partition((p: any) => p.branches([
    (e: any) => e.branch((b: any) => b.then([(w2: any) => w2.upsert((u: any) => u.exists([(q: any) => q.symbols({ y: Person9 })]))])),
    (e: any) => e.branch((b: any) => b),
  ]).then([(w2: any) => w2.update((u: any) => u.apply([(q: any) => q.symbols({ y: Person9 })]))]))])),
  "symbol 'y' of an update is bound by no term before it"); // bound in one branch only
  const book9 = store9.singleton("crm.TheBook");
  const ada9 = store9.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { ada: Person9, book: Book9 }).spec([(_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada) }),
    (_: any) => _.ada.name("Ada")])]))]))[0].get("ada");
  const orphaning = (t: any): any => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person9 })
    .spec([(_: any) => _.x.name("Ada")])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols({ x: Person9 })
    .spec([(_: any) => _.x.age(1n).knows([(e: any) => e.known(_.x)])])]).then([(w2: any) => w2.upsert((u2: any) => u2.exists([
    (p: any) => p.symbols({ o: Person9 }).spec([(_: any) => _.o.name("Orphan")])]))]))]))]);
  raises(ValueError, () => store9.transaction(orphaning), "a 'crm.Person' needs an owner"); // refused when the transaction ends
  assert(store9.member(ada9, "age") === null && ada9.knows.length === 0 && same([...store9.extent("crm.Person")], [ada9]));
  store9.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { bea: Person9, book: Book9 }).spec([(_: any) => _.book.contacts({ bea: (e: any) => e.contact(_.bea.name("Bea")) })])]))]));
  const knowing = store9.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: Person9 }).spec([(_: any) => _.x.name("Bea")])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person9 }).spec([(_: any) => _.x.knows([ada9])])]))]))])); // Ada by her path
  store9.transaction(() => {
    store9.crm.Book(book9, (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("ada")))); // Ada deleted
  });
  raises(LookupError, () => store9.transaction(knowing), "no object of the store has the path a spec refers to");
  assert(same([...book9.contacts].map((e: any) => e.contact.name.get()), ["Bea"]));
}

## TXN-10 · Schema objects in transactions: a module's install upserts its schemas, found by name, created where none is, a schema that existed extended when the transaction commits

In [ ]:
{
  const store10: any = new B.Store();
  const [Int10, OfObject] = [store10.Schemas.OfInt, store10.Schemas.OfObject];
  /** zoo's install ([CAP-115: schemas defined by idempotent installs](../../../mbse-basis/docs/capabilities/CAP-115.md#injection), Injection): its schemas, upserted by name. */
  const install = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("zoo-1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
      { Pet: S.OfObject, Likes: S.OfRelation }).spec([
      (_: any) => _.Pet.name("zoo.Pet").ref(true).properties({ name: S.OfStr, legs: S.OfInt }),
      (_: any) => _.Likes.name("zoo.Likes").links([(e: any) => e.link((k: any) => k.schema([_.Pet])).name("fan"),
        (e: any) => e.link((k: any) => k.schema([_.Pet])).name("idol")]),
      (_: any) => _.Pet.adjacencies({ likes: (a: any) => a.adjacency((j: any) => j.relation([_.Likes]).me("fan")) }),
    ])]))]);
  };
  const installed = store10.transaction(install)[0];
  const Pet = store10.registered("zoo.Pet");
  assert(installed.size === 2 && installed.get("Pet") === Pet && installed.get("Likes") === store10.registered("zoo.Likes"));
  assert(Pet.ref && same([...Pet.properties.keys()], ["name", "legs"]) && Pet.properties.get("legs").schema === Int10.as_schema());
  assert(same([...store10.registered("zoo.Likes").entry.links.keys()], ["fan", "idol"]) && Pet.adjacencies.get("likes").me === "fan");
  const again = store10.transaction(install)[0]; // found: as it was
  assert(again.get("Pet") === Pet && again.get("Likes") === installed.get("Likes") && same([...Pet.properties.keys()], ["name", "legs"]));
  assert(store10.module("zoo").schemas.get("Pet") === Pet);
  /** zoo.Pet's description and age, added to what `install` defined. */
  const aged = (txn: any): any => txn.uid("zoo-2").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { Pet: txn.Schemas.OfObject }).spec([(_: any) => _.Pet.name("zoo.Pet").description("a pet").properties({
      age: txn.Schemas.OfInt })])]))]);
  store10.transaction(aged);
  assert(Pet.description === "a pet" && same([...Pet.properties.keys()], ["name", "legs", "age"])); // extended, by its name
  const rex = store10.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { rex: Pet, tom: Pet }).spec([(_: any) => _.rex.name("Rex").legs(4n).likes([_.tom]), (_: any) => _.tom.name("Tom")])]))]));
  assert(rex[0].get("rex").name.get() === "Rex" && same([...rex[0].get("rex").likes].map((e: any) => e.idol.name.get()), ["Tom"])); // its data
  const named = store10.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { S: OfObject }).spec([(_: any) => _.S.ref(true)])]))]));
  assert(same(named.map((frame: any) => frame.get("S")), [Pet])); // the store's schemas, the framework's left out
  assert(same(store10.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ S: OfObject }).spec([(_: any) => _.S.properties({ legs: Int10 })])]))])).map((frame: any) =>
    frame.get("S").name), ["zoo.Pet"]));
  /** zoo.New, and a relation named as zoo.Pet is. */
  const clashing = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("zoo-3").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
      { New: S.OfObject, Clash: S.OfRelation }).spec([(_: any) => _.New.name("zoo.New").ref(true),
      (_: any) => _.Clash.name("zoo.Pet")])]))]);
  };
  raises(ValueError, () => store10.transaction(clashing), "schema 'zoo.Pet' is already registered"); // a relation can not take a schema's name
  assert(!store10.names().includes("zoo.New") && !store10.module("zoo").schemas.has("New"));
}

## TXN-11 · A transaction's imperative scope: builders write within it, all or nothing; outside one a write is refused, but an expression's, the store's making and a decoder's, which joins the transaction open

In [ ]:
{
  const store11: any = new B.Store(); // its root module written as it is made
  raises(ValueError, () => store11.Schemas.OfObject((o: any) => o.name("crm.Pet").ref(true)),
    "a store is written only within a transaction (store.transaction)");
  /** crm.Pet, counted, with a number. */
  const pets = (txn: any): any => txn.uid("pets/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Pet: txn.Schemas.OfObject })
    .spec([(_: any) => _.Pet.name("crm.Pet").ref(true).properties({ n: txn.Schemas.OfInt })])]))]);
  store11.transaction(pets);
  const Pet11 = store11.registered("crm.Pet");
  const rex11 = store11.transaction(() => {
    const rex = store11.crm.Pet((p: any) => p.n(1n));
    assert(store11.member(rex, "n") === 1n); // read within it
    return rex;
  });
  raises(ValueError, () => store11.crm.Pet(rex11, (p: any) => p.n(2n)), "written only within a transaction");
  raises(ValueError, () => store11.transaction(() => {
    store11.crm.Pet(rex11, (p: any) => p.n(3n));
    throw new ValueError("stop");
  }), "stop");
  assert(store11.member(rex11, "n") === 1n); // all or nothing
  const tx11 = store11.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ p: Pet11 }).spec([(_: any) => _.p.n(5n)])]))])); // an expression, built outside one (94)
  assert(store11.member(store11.transaction(tx11)[0].get("p"), "n") === 5n);
  const plain11 = Plain.ToPlain(store11)(Pet11, rex11);
  assert(store11.member(Plain.FromPlain(store11)(Pet11, plain11), "n") === 1n); // a decoder: a transaction of its own
  const other11: any = new B.Store();
  store11.transaction(() => {
    assert(store11.member(Plain.FromPlain(store11)(Pet11, plain11), "n") === 1n); // or the one open, joined
    raises(ValueError, () => other11.Schemas.OfObject((o: any) => o.name("crm.Pet").ref(true)),
      "written only within a transaction"); // another store's: not this one's
    const Other11 = other11.transaction(() => { // another store's transaction, open at once: its own (Decided, 99)
      const Other = other11.Schemas.OfObject((o: any) => o.name("crm.Other").ref(true).properties({ n: other11.Schemas.OfInt }));
      other11.crm.Other((o: any) => o.n(1n));
      return Other;
    });
    assert(other11.registered("crm.Other") === Other11); // committed when its own transaction ended
  });
  let kept11: any = null;
  raises(ValueError, () => store11.transaction(() => {
    store11.crm.Pet(rex11, (p: any) => p.n(7n));
    kept11 = other11.transaction(() => other11.crm.Other((o: any) => o.n(2n)));
    throw new ValueError("stop");
  }), "stop");
  assert(store11.member(rex11, "n") === 1n && other11.member(kept11, "n") === 2n); // each all or nothing on its own
  raises(ValueError, () => other11.transaction(() => {
    other11.crm.Other(kept11, (o: any) => o.n(3n));
    store11.transaction(() => store11.crm.Pet(rex11, (p: any) => p.n(8n)));
    throw new ValueError("stop");
  }), "stop");
  assert(store11.member(rex11, "n") === 8n && other11.member(kept11, "n") === 2n);
  raises(ValueError, () => store11.transaction(() => store11.transaction(() => null)),
    "a transaction is open: transactions do not nest");
}

## TXN-12 · Proposals: the store as a transaction has it, its writes held apart until it commits; within a transaction, a savepoint, a decoder's, which fails alone

In [ ]:
{
  const store12: any = new B.Store();
  /** crm.Pet, a singleton, crm.Rex, with a number. */
  const kennel = (txn: any): any => txn.uid("rex/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Pet: txn.Schemas.OfObject })
    .spec([(_: any) => _.Pet.name("crm.Pet").ref(true).singleton("crm.Rex").properties({ n: txn.Schemas.OfInt })])]))]);
  store12.transaction(kennel);
  const Pet12 = store12.registered("crm.Pet");
  const rex12 = store12.transaction(() => {
    const rex = store12.singleton("crm.Rex");
    store12.crm.Pet(rex, (p: any) => p.n(1n));
    return rex;
  });
  const plain12: any = Plain.ToPlain(store12)(Pet12, rex12);
  store12.transaction(() => {
    const proposals12 = store12._proposals; // the store as the transaction has it: its writes, apart until it commits
    assert(proposals12 instanceof Proposals.OfStore && instance_problems(proposals12, "OfStore").length === 0);
    assert(proposals12.schema("crm.Pet") === Pet12 && proposals12.registered("crm.Pet") === Pet12 && proposals12.name_of(Pet12) === "crm.Pet");
    assert(same(proposals12.names(), store12.names()) && !proposals12.framework(Pet12));
    assert(proposals12.singleton("crm.Rex") === rex12 && same(proposals12.extent("crm.Pet"), [rex12]));
    assert(proposals12.member(rex12, "n") === 1n && proposals12.builder("crm.Pet") !== null);
    store12.crm.Pet(rex12, (p: any) => p.n(2n));
    const state12 = Proxies._innerOf(rex12);
    assert(proposals12.values(state12) !== store12._committed.values(state12)); // written: the proposals' copy
    raises(ValueError, () => store12._joined(() => { // within a transaction, a savepoint: proposals over its proposals, dropped alone
      assert(store12._proposals.base === proposals12);
      store12.crm.Pet(rex12, (p: any) => p.n(3n));
      throw new ValueError("stop");
    }), "stop");
    assert(store12.member(rex12, "n") === 2n);
    const bad12 = Json.loads(Json.dumps(plain12).replace('"value": 1', '"value": "x"'));
    raises(DecodeError, () => Plain.FromPlain(store12)(Pet12, bad12),
      "expected int, got str"); // a decoder's: the transaction unaffected where it fails
    assert(store12.member(rex12, "n") === 2n);
    store12._joined(() => { // committed into the transaction's proposals
      store12.Schemas.OfObject(Pet12, (o: any) => o.description("a pet")); // a schema that existed: waits
      store12.Schemas.OfObject(Pet12, (o: any) => o.description("the pet")); // from what the one before made
      store12.crm.Pet(rex12, (p: any) => p.n(4n));
    });
    assert(Pet12.description === null && store12.member(rex12, "n") === 4n);
    assert(store12._committed.values(state12) !== proposals12.values(state12));
  });
  assert(Pet12.description === "the pet" && store12.member(rex12, "n") === 4n); // the transaction's commit
}

## TXN-13 · A transaction on the wire: its expression nested, each term where the one before links it, read back through the builders and executed; what a reader refuses

In [ ]:
{
  const store13: any = new B.Store();
  store13.transaction(crm);
  const [Person13, Book13] = [store13.registered("crm.Person"), store13.registered("crm.Book")];
  const named13 = (w: any) => w.update((d: any) => d.apply([(p: any) => p.symbols({ ada: Person13 }).spec([(_: any) => _.ada.name("Ada L.")])]));
  const dropped13 = (w: any) => w.reduce((r: any) => r.symbols([(c: any) => c.pattern([(p: any) => p.symbols({ book: Book13 })]).symbols(["book"])]));
  const tx13 = store13.Mutations.OfTransaction((t: any) => t.uid("6f1c4e2a").mutations([(mu: any) => mu.upsert((u: any) => u.key("k").exists([
    (p: any) => p.symbols({ book: Book13, ada: Person13 }).spec([(_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada) }),
      (_: any) => _.ada.name("Ada")]),
  ]).then([(w: any) => w.partition((q: any) => q.first(true).branches([(e: any) => e.branch((b: any) => b.then([named13]))])
    .none([dropped13]))]))]));
  const wire13: any = Plain.ToPlain(store13).Transaction(tx13); // its expression, nested: each term where the one before links it
  assert(same([...wire13.keys()], ["uid", "mutations"]) && same(wire13.get("mutations").map((term: any) => [...term.keys()][0]), ["upsert"]));
  const upsert13 = wire13.get("mutations")[0].get("upsert");
  assert(same([...upsert13.keys()], ["key", "exists", "then"])); // its properties, then its adjacencies, in its schema's order
  const pattern13 = upsert13.get("exists")[0];
  assert(same([...pattern13.keys()], ["symbols", "spec"]) && same([...pattern13.get("symbols").keys()], ["book", "ada"]));
  assert(equal(pattern13.get("symbols").get("ada"), map({ schema: [{ $ref: Plain.ToPlain(store13)(
    store13.Paths.OfRoot, Paths.of(Person13)) }] }))); // a schema by its path
  assert(equal(pattern13.get("spec"), [...[...[...tx13.mutations][0].term.exists][0].pattern.spec]));
  const partition13 = upsert13.get("then")[0].get("partition");
  assert(partition13.get("first") === true && same([...partition13.get("branches")[0].keys()], ["then"])
    && same([...partition13.get("none")[0].keys()], ["reduce"]));
  const back13 = Plain.FromPlain(store13).Transaction(Json.loads(Json.dumps(wire13))); // built anew, through the builders
  assert(back13 !== tx13 && equal(Plain.ToPlain(store13).Transaction(back13), wire13));
  const frames13 = store13.transaction(back13);
  assert(equal(frames13.map((frame: any) => [frame.get("ada").name.get(), frame.get("book") === store13.singleton("crm.TheBook")]),
    [["Ada L.", true]]));
  const Int13 = store13.Schemas.OfInt;
  const varied13 = store13.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.variables([
    (p: any) => p.variables({ v: (e: any) => e.variable((v: any) => v.schema([store13.Schemas.OfUnion((u: any) => u.branches([
      (b: any) => b.name("a").schema([Int13])]))]))})]))]));
  const schema13 = Plain.ToPlain(store13).Transaction(varied13).get("mutations")[0].get("map").get("variables")[0].get("variables").get("v").get("schema");
  assert(same([...schema13[0].keys()], ["Schemas.OfUnion"])); // several schemas fill the link: tagged with its schema's name
  assert(schema13[0].get("Schemas.OfUnion").get("branches")[0].get("name") === "a"); // an entry with a property: its links and properties
  assert(equal(Plain.ToPlain(store13).Transaction(Plain.FromPlain(store13).Transaction(
    Plain.ToPlain(store13).Transaction(varied13))), Plain.ToPlain(store13).Transaction(varied13)));
  const read13 = Plain.FromPlain(store13).Transaction;
  const upsert = (body: unknown) => map({ mutations: [{ upsert: body }] });
  for (const [plain13, message13] of [
    [[], "expected an object of Mutations.OfTransaction, a mapping of its members"],
    [map({ nope: 1n }), "'nope' is not a member of Mutations.OfTransaction"],
    [map({ mutations: {} }), "expected a list of what the entries link"],
    [map({ mutations: [{ nope: {} }] }), "expected one of the kinds map, reduce, partition, update, upsert, as {kind: object}"],
    [upsert({ exists: [{ symbols: [] }] }), "expected a mapping of each entry's name to what it links"],
    [upsert({ exists: [{ symbols: { x: { schema: [{ $ref: { singleton: "crm.Nope", next: { target: {} } } }] } } }] }),
      "no object at this path"],
    [upsert({ exists: [{ variables: { v: { schema: [{ "Mutations.OfMap": {} }] } } }] }),
      "expected an object of one of Schemas.OfScalar, Schemas.OfObject, Schemas.OfUnion"],
    [upsert({ exists: [{ variables: { v: { schema: [{ "Schemas.OfUnion": { branches: {} } }] } } }] }), "expected a list of entries"],
    [upsert({ exists: [{ variables: { v: { schema: [{ "Schemas.OfUnion": { branches: [1n] } }] } } }] }),
      "expected an entry, a mapping of its links and properties"],
    [upsert({ exists: [{ variables: { v: { schema: [{ "Schemas.OfUnion": { branches: [{ x: 1n }] } }] } } }] }),
      "'x' is not a link or property of the entry"],
  ] as [unknown, string][]) {
    raises(DecodeError, () => read13(plain13), message13);
  }
  const book13 = store13.singleton("crm.TheBook");
  const typed13 = (symbol: string) => ({ schema: [pattern13.get("symbols").get(symbol).get("schema")[0]] });
  const adjacency13 = (name: string, ...verbs: unknown[]) => ({ as_adjacency: { name, mutations: verbs } });
  const link13 = (name: string, verb: unknown) => ({ as_link: { name, mutations: [verb] } });
  const since13 = (verb: unknown) => ({ as_property: { name: "since", mutations: [verb] } });
  const knowing13 = read13(upsert({ exists: [{ symbols: { book: typed13("book"), ada: typed13("ada") }, spec: [
    { book: [adjacency13("contacts", { insert: { entries: [{ mutations: [
      link13("contact", { set: { ref: { symbol: "ada" } } }), { as_property: { name: "key", mutations: [
        { set: { value: "ada" } }] } }] }] } })] },
    { ada: [adjacency13("knows", { insert: { entries: [{ mutations: [
      link13("known", { set: { ref: { symbol: "ada" } } }), since13({ clear: {} })] }] } })] }] }] }));
  const ada13 = store13.transaction(knowing13)[0].get("ada"); // a spec read from the wire may clear an entry's property
  assert(equal([...ada13.knows].map((e: any) => [e.known === ada13, e.since.get()]), [[true, null]])); // inserted without it
  const removing13 = read13(upsert({ exists: [{ symbols: { book: typed13("book") }, spec: [
    { book: [adjacency13("contacts", { remove: { entries: [] } })] }] }] }));
  store13.transaction(removing13); // removing no entry: nothing removed
  assert(equal([...book13.contacts].map((e: any) => e.key.get()), ["ada"]));
  const emptying13 = read13(upsert({ exists: [{ symbols: { book: typed13("book") }, spec: [
    { book: [adjacency13("contacts", { clear: {} })] }] }] }));
  store13.transaction(emptying13); // read, as any spec's verb, and run
  assert(book13.contacts.length === 0);
  for (const verb13 of ["insert", "remove"]) { // an entry inserted or removed
    const clearing13 = read13(upsert({ exists: [{ symbols: { book: typed13("book") }, spec: [{ book: [adjacency13(
      "contacts", { [verb13]: { entries: [{ mutations: [link13("contact", { clear: {} })] }] } })] }] }] }));
    raises(ValueError, () => store13.transaction(clearing13),
      "a spec's entry clears a link: every link of an entry but its object's own is set");
  }
  raises(DecodeError, () => read13(upsert({ exists: [{ symbols: { book: typed13("book") }, spec: [
    { book: [adjacency13("contacts", { replace: { entries: [] } })] }] }] })), // an adjacency inserts, removes or clears
  "expected a spec's mutations");
  const branch13 = store13.Mutations.OfBranch((b: any) => b);
  const twice13 = store13.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.partition((q: any) => q.branches([branch13]).then([
    (w: any) => w.partition((r: any) => r.branches([branch13]))]))])); // one branch, two partitions
  raises(ValueError, () => Plain.ToPlain(store13).Transaction(twice13),
    "an expression links one of its terms from two places: it is written as a tree");
  const given13 = store13.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ b: References.of(store13, book13) }).spec([(_: any) => _.b.contacts((a: any) => a.clear())])]))]));
  const sent13: any = Plain.ToPlain(store13).Transaction(given13); // a symbol given a reference: written in place of its schema
  const reference13 = Plain.ToPlain(store13)(store13.References.OfObject, References.of(store13, book13));
  assert(equal(sent13.get("mutations")[0].get("map").get("symbols")[0].get("symbols"), map({ b: { reference: reference13 } })));
  assert(equal(reference13, map({ path: { singleton: "crm.TheBook", next: { target: {} } } }))); // as any References.OfObject
  const got13 = read13(Json.loads(Json.dumps(sent13)));
  assert(equal(Plain.ToPlain(store13).Transaction(got13), sent13)
    && equal(store13.transaction(got13), [new Map([["b", book13]])]));
  sent13.get("mutations")[0].get("map").get("symbols")[0].get("symbols").get("b").get("reference").get("path")
    .set("singleton", "crm.Nope");
  raises(DecodeError, () => read13(sent13), // its spec read against the schema its path implies, through the schemas
    "symbol 'b' is given a reference whose path starts from 'crm.Nope', which no schema declares a singleton");
}

## TXN-14 · Removing entries through a spec: an adjacency's remove, each entry agreeing with the one given, and clear; a spec holds where what it removes is absent, so run again it writes nothing; an owning entry removed deletes what it owned

In [ ]:
{
  const store14: any = new B.Store();
  store14.transaction(crm);
  const [Person14, Book14] = [store14.registered("crm.Person"), store14.registered("crm.Book")];
  const book14 = store14.singleton("crm.TheBook");
  const people14 = { book: Book14, ada: Person14, bob: Person14 };
  /** An upsert of Ada and Bob, each found by the book's entry that owns it, and `specs` written over them. */
  const upserted = (...specs: unknown[]): any => store14.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) =>
    u.exists([(p: any) => p.symbols(people14).spec([(_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada),
      bob: (e: any) => e.contact(_.bob) }), ...specs])]))]))[0];
  const known = (person: any): unknown[] => [...person.knows].map((e: any) => [e.known.name.get(), e.since.get()]);
  const snapshot = (): string => Json.dumps(Plain.ToPlain(store14).Snapshot() as never);
  const fixture14 = upserted((_: any) => _.ada.name("Ada").knows([
    (e: any) => e.known(_.bob).since(2020n), (e: any) => e.known(_.bob).since(2021n), (e: any) => e.known(_.ada).since(2022n)]),
  (_: any) => _.bob.name("Bob").knows([_.ada]));
  const [ada14, bob14] = [fixture14.get("ada"), fixture14.get("bob")];
  let snapshot14 = snapshot();
  upserted((_: any) => _.ada.knows((a: any) => a.remove((e: any) => e.known(_.bob).since(1999n)))); // agreeing with none: nothing
  assert(equal(known(ada14), [["Bob", 2020n], ["Bob", 2021n], ["Ada", 2022n]]) && snapshot() === snapshot14);
  const unknown = (_: any) => _.ada.knows((a: any) => a.remove((e: any) => e.known(_.bob))); // by its link: every link it gives agrees
  upserted(unknown);
  assert(equal(known(ada14), [["Ada", 2022n]]));
  snapshot14 = snapshot();
  upserted(unknown); // run again: what it removes is absent, so its spec holds and nothing is written
  assert(snapshot() === snapshot14);
  const mapped14 = (...specs: unknown[]): string[] => store14.transaction((t: any) => t.mutations([(mu: any) => mu.map(
    (m: any) => m.symbols([(p: any) => p.symbols({ x: Person14, y: Person14 }).spec([(_: any) => _.y.name("Bob"), ...specs])]))]))
    .map((f: any) => f.get("x").name.get()).sort();
  assert(equal(mapped14((_: any) => _.x.knows((a: any) => a.remove((e: any) => e.known(_.y)))), ["Ada", "Bob"])); // where it is absent
  assert(equal(mapped14((_: any) => _.x.knows((a: any) => a.remove((e: any) => e.known(_.x)))), ["Bob"]));
  assert(equal(mapped14((_: any) => _.x.knows((a: any) => a.clear())), [])); // none held: each knows someone
  upserted((_: any) => _.ada.knows((a: any) => a.insert((e: any) => e.known(_.bob).since(2023n)).remove((e: any) => e.since(2022n))));
  assert(equal(known(ada14), [["Bob", 2023n]])); // by a property
  store14.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person14 }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
    { x: Person14 }).spec([(_: any) => _.x.knows((a: any) => a.clear())])]))]))])); // an update clears: every entry removed
  assert(equal(known(bob14), []) && equal(known(ada14), [["Bob", 2023n]]));
  assert(equal(mapped14((_: any) => _.x.knows((a: any) => a.clear())), ["Bob"]));
  upserted((_: any) => _.ada.knows((a: any) => a.clear().insert((e: any) => e.known(_.bob))));
  assert(equal(known(ada14), [["Bob", null]])); // cleared, then inserted, in the order the spec gives them
  store14.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ book: Book14 }).spec([
    (_: any) => _.book.contacts((a: any) => a.remove((e: any) => e.key("bob")))])]))])); // an owning entry, by its key
  assert(equal([...book14.contacts].map((e: any) => e.key.get()), ["ada"])
    && equal([...store14.extent("crm.Person")].map((p: any) => p.name.get()), ["Ada"]));
  assert(equal(known(ada14), [])); // Bob deleted with it, and every entry linking him
}